# FX Backtester — full report

The whole framework end to end: load the 1-second 2024 EURUSD file → resample to
5-minute bars → run the reference `SmaCrossoverStrategy(20/50)` through the engine
→ evaluate (normal + adversarial metrics) → the three plots.

See [`docs.md`](../docs.md) for the API and [`regression.md`](../regression.md)
for the §6 regression.


In [1]:
import sys; sys.path.insert(0, "..")
%matplotlib inline

from lib.data import load_1s_data, resample
from lib.engine import Engine
from lib.strategies import SmaCrossoverStrategy
from lib.evaluate import plot_equity, plot_monthly, plot_mc_drawdown

df      = load_1s_data("../data/EURUSD_1s_2024.csv", verbose=False)
engine  = Engine(resample(df, "5m"), df)
trades  = engine.backtest(SmaCrossoverStrategy(fast_n=20, slow_n=50, sl_pips=10, tp_pips=20))

report  = engine.evaluate(trades, starting_balance=10_000, pip_value=1.0, seed=0)
print(f"{trades.height:,} trades   |   verdict: {report['verdict']}")


1,714 trades   |   verdict: unprofitable


## Normal metrics (§3.1)


In [2]:
for k, v in report["summary"].items():
    print(f"  {k:20s} {v}")


  n_trades             1714
  win_rate             32.497082847141186
  expectancy_pips      -0.264352392065085
  profit_factor        0.9353600776079107
  avg_win_pips         11.77109515260319
  avg_loss_pips        -6.116579406631358
  total_pips           -453.0999999995556
  total_return_pct     -4.530999999995556
  max_drawdown_pct     -8.933999999996649
  max_drawdown_date    2024-10-29 11:05:00+00:00
  sharpe               -0.9641451487778472
  sortino              -1.5400752046938082
  mar                  -0.5079609846585421


In [3]:
report["monthly"]


shape: (12, 5)
┌────────────┬────────┬────────┬──────────┬───────────┐
│ month      ┆ pips   ┆ pnl    ┆ n_trades ┆ win_rate  │
│ ---        ┆ ---    ┆ ---    ┆ ---      ┆ ---       │
│ date       ┆ f64    ┆ f64    ┆ u32      ┆ f64       │
╞════════════╪════════╪════════╪══════════╪═══════════╡
│ 2024-01-01 ┆ -219.0 ┆ -219.0 ┆ 152      ┆ 27.631579 │
│ 2024-02-01 ┆ 17.0   ┆ 17.0   ┆ 132      ┆ 35.606061 │
│ 2024-03-01 ┆ -32.2  ┆ -32.2  ┆ 148      ┆ 30.405405 │
│ 2024-04-01 ┆ -118.2 ┆ -118.2 ┆ 135      ┆ 34.074074 │
│ 2024-05-01 ┆ -12.3  ┆ -12.3  ┆ 133      ┆ 33.082707 │
│ …          ┆ …      ┆ …      ┆ …        ┆ …         │
│ 2024-08-01 ┆ -57.0  ┆ -57.0  ┆ 155      ┆ 31.612903 │
│ 2024-09-01 ┆ -44.8  ┆ -44.8  ┆ 137      ┆ 35.036496 │
│ 2024-10-01 ┆ -146.2 ┆ -146.2 ┆ 165      ┆ 27.878788 │
│ 2024-11-01 ┆ 290.9  ┆ 290.9  ┆ 121      ┆ 42.975207 │
│ 2024-12-01 ┆ 42.9   ┆ 42.9   ┆ 149      ┆ 32.885906 │
└────────────┴────────┴────────┴──────────┴───────────┘

## Adversarial metrics (§3.2)


In [4]:
adv = report["adversarial"]

print("gross vs net:")
for k, v in adv["gross_vs_net"].items():
    print(f"  {k:20s} {v}")

print("\nexcluding the best month:")
for k, v in adv["ex_best_month"].items():
    print(f"  {k:22s} {v}")

print("\nexcluding the best N% of trades:")
for row in adv["ex_best_trades"]:
    print(f"  drop top {row['pct']:>4.0%} ({row['n_dropped']:>2d} trades)  ->  {row['pips']:>8.1f} pips")

print("\nsample size:", adv["sample_size"])


gross vs net:
  net_pips             -453.0999999995556
  spread_paid_pips     597.9000000000301
  gross_pips           144.80000000047448
  net_pnl              -453.0999999995556
  gross_pnl            144.80000000047448
  spread_paid_pnl      597.9000000000301
  edge_assessment      edge existed, costs ate it — gross positive but net not

excluding the best month:
  dropped_month          2024-11-01
  dropped_month_pips     290.900000000054
  pips                   -743.9999999996094
  pnl                    -743.9999999996094
  full_pips              -453.09999999955534

excluding the best N% of trades:
  drop top   1% (18 trades)  ->    -813.1 pips
  drop top   5% (86 trades)  ->   -2173.1 pips

sample size: {'n_trades': 1714, 'min_trades': 100, 'low_sample': False, 'warning': None}


In [5]:
bs = adv["bootstrap_ci"]
print(f"bootstrap ({bs['n_resamples']:,} resamples, {bs['confidence']:.0%} CI, seed {bs['seed']}):")
for stat in ("win_rate", "expectancy_pips"):
    s = bs[stat]
    print(f"  {stat:16s} observed {s['observed']:.3f}   CI [{s['ci_low']:.3f}, {s['ci_high']:.3f}]")


bootstrap (10,000 resamples, 95% CI, seed 0):
  win_rate         observed 32.497   CI [30.222, 34.714]
  expectancy_pips  observed -0.264   CI [-0.744, 0.210]


In [6]:
mc = adv["mc_drawdown"]
print(f"Monte-Carlo trade-order shuffle ({mc['n_shuffles']:,} shuffles, seed {mc['seed']}):")
print(f"  observed max drawdown   {mc['observed_max_drawdown_pct']:.2f}%")
print(f"  shuffled  median        {mc['median']:.2f}%")
print(f"  shuffled  p95 (worse)   {mc['p95']:.2f}%")
print(f"  shuffled  worst         {mc['worst']:.2f}%")
print(f"  reorderings worse       {mc['percentile_rank']:.0f}%")
print(f"  fragile                 {mc['fragile']}")
print(f"\n  {mc['note']}")


Monte-Carlo trade-order shuffle (10,000 shuffles, seed 0):
  observed max drawdown   -8.93%
  shuffled  median        -6.84%
  shuffled  p95 (worse)   -9.04%
  shuffled  worst         -12.80%
  reorderings worse       6%
  fragile                 False

  This tests sequence risk on the trades you already have — it cannot detect a missing edge, and a good shuffle distribution is not validation of the strategy.


## Plots


In [7]:
plot_equity(report)


In [8]:
plot_monthly(report)


In [9]:
plot_mc_drawdown(report["adversarial"]["mc_drawdown"])


## Read

`SmaCrossoverStrategy(20/50, sl 10 / tp 20)` on 2024 EURUSD: a **losing** strategy
after costs. Gross P&L (before spread) is marginally positive, so the spread is
what tips it negative — but the bootstrap CI on expectancy straddles zero, so
there isn't a statistically real edge either way. The framework's job is to make
that legible; it does.
